# 🔧 LangGraph `ToolNode` — Complete Implementation Guide

> **What is `ToolNode`?**  
> `ToolNode` is a prebuilt LangGraph node that executes tools inside a graph workflow.  
> Unlike `create_agent` (which handles everything end-to-end), `ToolNode` gives you **fine-grained control** over tool execution — making it the right choice for production-grade, custom agent architectures.

### What you'll learn:
- ✅ Basic `ToolNode` setup with a real graph
- ✅ Tools that return **strings**, **objects (dicts)**, and **Commands**
- ✅ Parallel tool execution
- ✅ Error handling strategies
- ✅ A complete, runnable agent built on `ToolNode`

---
## Cell 1 — Install & Imports

In [ ]:
%pip install langchain==1.0.0 langgraph==1.0.3 langchain-openai python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langchain.messages import ToolMessage
from langchain.tools import tool, ToolRuntime

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Command

load_dotenv()
print("✅ All imports successful!")

---
## Cell 2 — LLM Setup

In [ ]:
llm = ChatOpenAI(
    model=os.getenv("MODEL"),
    base_url=os.getenv("API_URL"),
    api_key=os.getenv("API_KEY"),
    temperature=0,
)
print("✅ LLM configured!")

---
## Part 1 — Basic `ToolNode` Setup

This is the **simplest possible** graph with a `ToolNode`.  
The LLM decides which tool to call → `ToolNode` executes it → LLM summarizes.

```
START → llm_node → tool_node → llm_node → END
              ↑_________________↑   (loop until no more tool calls)
```

In [ ]:
# --- Define Tools ---

@tool
def search(query: str) -> str:
    """Search for information on a topic."""
    # Simulated search result
    return f"Top result for '{query}': This is a simulated search result with relevant information."

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic math expression safely."""
    try:
        # Safe eval — only allow simple arithmetic
        allowed = set('0123456789+-*/(). ')
        if all(c in allowed for c in expression):
            result = eval(expression)
            return f"{expression} = {result}"
        else:
            return "Error: Only basic arithmetic is allowed."
    except Exception as e:
        return f"Error evaluating expression: {e}"

print("✅ Tools defined!")

In [ ]:
# --- Create ToolNode ---

tools = [search, calculator]
tool_node = ToolNode(tools)          # Handles parallel execution & errors automatically

llm_with_tools = llm.bind_tools(tools)   # Tell the LLM which tools are available

print("✅ ToolNode created!")
print(f"   Tools registered: {[t.name for t in tools]}")

In [ ]:
# --- Build the Graph ---

def llm_node(state: MessagesState):
    """Node that calls the LLM and returns its response."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: MessagesState):
    """Route to tools if LLM made tool calls, otherwise end."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"     # LLM wants to call a tool
    return END             # LLM is done, return final answer

# Wire up the graph
builder = StateGraph(MessagesState)
builder.add_node("llm", llm_node)
builder.add_node("tools", tool_node)        # ← ToolNode plugs in here

builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", should_continue)   # llm → tools OR end
builder.add_edge("tools", "llm")            # after tools, always go back to LLM

graph = builder.compile()
print("✅ Graph compiled!")

In [ ]:
# --- Run the Basic Graph ---

print("▶️  Running basic ToolNode graph...\n")

result = graph.invoke({
    "messages": [HumanMessage(content="What is 128 * 256? Also search for LangGraph.")]
})

print("🤖 Final Answer:", result["messages"][-1].content)
print(f"\n📊 Total messages in state: {len(result['messages'])}")
for i, msg in enumerate(result["messages"]):
    print(f"   [{i}] {type(msg).__name__}: {str(msg.content)[:80]}...")

---
## Part 2 — Tool Return Values

Tools can return **3 different types**, each with different behavior:

| Return Type | Use Case |
|---|---|
| `str` | Plain text result the model reads |
| `dict` | Structured data for model to reason over |
| `Command` | Mutate graph state directly |

### 2a — Return a String (most common)

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    # Returns a plain human-readable string
    # → Converted to a ToolMessage, model reads the text
    return f"It is currently sunny in {city} with a temperature of 28°C."

# Quick test (outside graph)
print("Tool output (string):", get_weather.invoke({"city": "Mumbai"}))
print("\n✅ String return type — the model reads this text directly.")

### 2b — Return an Object (dict)

In [ ]:
@tool
def get_weather_data(city: str) -> dict:
    """Get structured weather data for a city."""
    # Returns a dict — model can reason over individual fields
    # → Serialized to JSON in the ToolMessage
    return {
        "city": city,
        "temperature_c": 28,
        "temperature_f": 82,
        "conditions": "sunny",
        "humidity_pct": 65,
        "wind_kph": 12,
    }

# Quick test (outside graph)
import json
print("Tool output (dict):")
print(json.dumps(get_weather_data.invoke({"city": "Mumbai"}), indent=2))
print("\n✅ Dict return type — model inspects specific fields for reasoning.")

### 2c — Return a Command (state mutation)

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# Custom state that holds extra fields beyond just messages
class AppState(TypedDict):
    messages: Annotated[list, add_messages]
    preferred_language: str              # ← extra state field a tool can set
    user_name: str                       # ← another custom field

@tool
def set_language(language: str, runtime: ToolRuntime) -> Command:
    """Set the preferred response language for the conversation."""
    # Command lets the tool WRITE to graph state
    # runtime.tool_call_id links the ToolMessage back to the tool call
    return Command(
        update={
            "preferred_language": language,
            "messages": [
                ToolMessage(
                    content=f"✅ Language preference updated to: {language}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

@tool
def set_user_name(name: str, runtime: ToolRuntime) -> Command:
    """Save the user's name for personalized responses."""
    return Command(
        update={
            "user_name": name,
            "messages": [
                ToolMessage(
                    content=f"✅ Name saved: {name}. I'll remember you!",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

print("✅ Command-returning tools defined!")
print("   These tools write directly to graph state when called.")

In [ ]:
# --- Build a graph that uses Command-returning tools ---

state_tools = [set_language, set_user_name, get_weather]
state_tool_node = ToolNode(state_tools)
state_llm = llm.bind_tools(state_tools)

def state_llm_node(state: AppState):
    response = state_llm.invoke(state["messages"])
    return {"messages": [response]}

def state_should_continue(state: AppState):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

state_builder = StateGraph(AppState)
state_builder.add_node("llm", state_llm_node)
state_builder.add_node("tools", state_tool_node)
state_builder.add_edge(START, "llm")
state_builder.add_conditional_edges("llm", state_should_continue)
state_builder.add_edge("tools", "llm")

state_graph = state_builder.compile()
print("✅ State-mutating graph compiled!")

In [ ]:
# --- Run it and observe state mutation ---

print("▶️  Running Command-returning tools demo...\n")

result = state_graph.invoke({
    "messages": [HumanMessage(content="My name is Arjun. Set my language to Hindi. Also get weather for Mumbai.")],
    "preferred_language": "",     # initial empty state
    "user_name": "",
})

print("🤖 Final Answer:", result["messages"][-1].content)
print()
print("📦 Final Graph State:")
print(f"   preferred_language : {result['preferred_language']}")
print(f"   user_name          : {result['user_name']}")
print()
print("✅ The tools wrote to state directly via Command!")

---
## Part 3 — Parallel Tool Execution

`ToolNode` runs multiple tool calls **in parallel automatically** when the LLM requests more than one tool in a single response.  
No extra configuration needed — it's built in.

In [ ]:
import time

@tool
def get_stock_price(ticker: str) -> dict:
    """Get the current stock price for a ticker symbol."""
    time.sleep(0.3)   # simulate network latency
    prices = {"AAPL": 189.50, "GOOGL": 175.20, "MSFT": 420.10, "AMZN": 195.80}
    price = prices.get(ticker.upper(), 100.00)
    return {"ticker": ticker.upper(), "price_usd": price, "currency": "USD"}

@tool
def get_company_info(ticker: str) -> dict:
    """Get basic company information for a ticker symbol."""
    time.sleep(0.3)   # simulate network latency
    info = {
        "AAPL":  {"name": "Apple Inc.",      "sector": "Technology", "employees": 161000},
        "GOOGL": {"name": "Alphabet Inc.",   "sector": "Technology", "employees": 182000},
        "MSFT":  {"name": "Microsoft Corp.", "sector": "Technology", "employees": 221000},
        "AMZN":  {"name": "Amazon.com Inc.", "sector": "Retail/Cloud","employees": 1540000},
    }
    return info.get(ticker.upper(), {"name": "Unknown", "sector": "N/A"})

print("✅ Stock tools defined (each simulates 0.3s network delay)")

In [ ]:
# Build a graph with parallel-capable ToolNode
parallel_tools = [get_stock_price, get_company_info]
parallel_tool_node = ToolNode(parallel_tools)       # parallel execution is automatic
parallel_llm = llm.bind_tools(parallel_tools)

def parallel_llm_node(state: MessagesState):
    return {"messages": [parallel_llm.invoke(state["messages"])]}

def parallel_should_continue(state: MessagesState):
    last = state["messages"][-1]
    return "tools" if (hasattr(last, "tool_calls") and last.tool_calls) else END

p_builder = StateGraph(MessagesState)
p_builder.add_node("llm", parallel_llm_node)
p_builder.add_node("tools", parallel_tool_node)
p_builder.add_edge(START, "llm")
p_builder.add_conditional_edges("llm", parallel_should_continue)
p_builder.add_edge("tools", "llm")
parallel_graph = p_builder.compile()

print("✅ Parallel ToolNode graph compiled!")

In [ ]:
print("▶️  Asking for both price AND company info — ToolNode runs them in parallel...\n")

start_time = time.time()
result = parallel_graph.invoke({
    "messages": [HumanMessage(content="Get the stock price AND company info for Apple (AAPL).")]
})
elapsed = time.time() - start_time

print("🤖 Final Answer:", result["messages"][-1].content)
print(f"\n⏱️  Total time: {elapsed:.2f}s")
print("   (Both tools ran in parallel — not sequentially!")
print("    Sequential would be ~0.6s. Parallel is ~0.3s.)")

---
## Part 4 — Error Handling

`ToolNode` has built-in error handling.  
By default, errors are caught and returned as error messages to the LLM (so it can recover gracefully).

In [ ]:
@tool
def divide(a: float, b: float) -> float:
    """Divide two numbers. Will fail if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero!")
    return a / b

@tool
def fetch_data(source: str) -> str:
    """Fetch data from a source. May fail for unknown sources."""
    known_sources = ["database", "api", "cache"]
    if source not in known_sources:
        raise ConnectionError(f"Unknown source: '{source}'. Valid: {known_sources}")
    return f"Data successfully fetched from {source}."

print("✅ Error-prone tools defined!")

In [ ]:
# --- Strategy 1: Default behavior (handle_tool_errors=True) ---
# Errors are caught and returned to the LLM as a friendly error message
# The LLM can then retry or explain what went wrong

error_tools = [divide, fetch_data]

# handle_tool_errors=True is the DEFAULT — LLM sees the error and can recover
safe_tool_node = ToolNode(error_tools, handle_tool_errors=True)

error_llm = llm.bind_tools(error_tools)

def error_llm_node(state: MessagesState):
    return {"messages": [error_llm.invoke(state["messages"])]}

def error_should_continue(state: MessagesState):
    last = state["messages"][-1]
    return "tools" if (hasattr(last, "tool_calls") and last.tool_calls) else END

e_builder = StateGraph(MessagesState)
e_builder.add_node("llm", error_llm_node)
e_builder.add_node("tools", safe_tool_node)
e_builder.add_edge(START, "llm")
e_builder.add_conditional_edges("llm", error_should_continue)
e_builder.add_edge("tools", "llm")
error_graph = e_builder.compile()

print("✅ Error-handling graph compiled!")
print("   handle_tool_errors=True → errors returned to LLM, not raised")

In [ ]:
# --- Run with a query that will trigger an error ---

print("▶️  Triggering a tool error — watch the LLM recover gracefully...\n")

result = error_graph.invoke({
    "messages": [HumanMessage(content="Please divide 10 by 0, and also fetch data from 'unknown_source'.")]
})

print("🤖 LLM Response (after seeing errors):")
print(result["messages"][-1].content)
print()
print("✅ The LLM received the error messages and responded gracefully!")

In [ ]:
# --- Strategy 2: Custom error message ---
# Pass a string to customize what the LLM sees on error

custom_error_node = ToolNode(
    error_tools,
    handle_tool_errors="⚠️ A tool failed. Please try a different approach or inform the user."
)
print("✅ Custom error message ToolNode created!")
print('   On any error → LLM sees: "⚠️ A tool failed. Please try a different approach..."')

In [ ]:
# --- Strategy 3: Disable error handling (raise immediately) ---
# Use this when you want errors to bubble up to your application code

strict_tool_node = ToolNode(error_tools, handle_tool_errors=False)
print("✅ Strict ToolNode created (handle_tool_errors=False)")
print("   Any tool error will raise an exception immediately.")
print("   Use this when YOUR code needs to handle errors, not the LLM.")

---
## Part 5 — Complete Production-Grade Agent

Putting it all together: a **customer support agent** that uses all three tool return types, parallel execution, and graceful error handling.

In [ ]:
# === Customer Support Agent using ToolNode ===

# -- State --
class SupportState(TypedDict):
    messages: Annotated[list, add_messages]
    user_name: str
    account_tier: str     # e.g. "free", "pro", "enterprise"

# -- Tools --

@tool
def lookup_order(order_id: str) -> dict:
    """Look up the status of a customer order by order ID."""
    orders = {
        "ORD-001": {"status": "delivered",   "item": "Laptop",     "date": "2025-03-15"},
        "ORD-002": {"status": "in_transit",  "item": "Headphones", "date": "2025-03-30"},
        "ORD-003": {"status": "processing",  "item": "Keyboard",   "date": "2025-04-01"},
    }
    if order_id not in orders:
        raise ValueError(f"Order {order_id} not found in system.")
    return orders[order_id]

@tool
def check_refund_eligibility(order_id: str) -> str:
    """Check if an order is eligible for a refund."""
    eligible = {"ORD-001": True, "ORD-002": False, "ORD-003": True}
    if order_id not in eligible:
        return f"Order {order_id} not found."
    return f"Order {order_id} is {'eligible' if eligible[order_id] else 'NOT eligible'} for a refund."

@tool
def escalate_to_human(reason: str, runtime: ToolRuntime) -> Command:
    """Escalate the issue to a human agent."""
    # This tool updates state to signal escalation
    return Command(
        update={
            "account_tier": "escalated",
            "messages": [
                ToolMessage(
                    content=f"🔔 Escalated to human agent. Reason: {reason}. A human will contact you within 24 hours.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

support_tools = [lookup_order, check_refund_eligibility, escalate_to_human]
support_tool_node = ToolNode(support_tools, handle_tool_errors=True)
support_llm = llm.bind_tools(support_tools)

print("✅ Support agent tools defined!")

In [ ]:
SUPPORT_SYSTEM_PROMPT = """
You are a helpful customer support agent. You have access to tools to:
- Look up order status
- Check refund eligibility
- Escalate to a human agent if needed

Always be polite and empathetic. Use tools to get accurate information before responding.
If multiple pieces of information are needed, call the relevant tools together.
"""

from langchain_core.messages import SystemMessage

def support_llm_node(state: SupportState):
    messages = [SystemMessage(content=SUPPORT_SYSTEM_PROMPT)] + state["messages"]
    response = support_llm.invoke(messages)
    return {"messages": [response]}

def support_should_continue(state: SupportState):
    last = state["messages"][-1]
    return "tools" if (hasattr(last, "tool_calls") and last.tool_calls) else END

s_builder = StateGraph(SupportState)
s_builder.add_node("llm", support_llm_node)
s_builder.add_node("tools", support_tool_node)
s_builder.add_edge(START, "llm")
s_builder.add_conditional_edges("llm", support_should_continue)
s_builder.add_edge("tools", "llm")
support_graph = s_builder.compile()

print("✅ Full support agent graph compiled!")

In [ ]:
# --- Test 1: Order lookup + refund check (parallel tools) ---

print("=" * 60)
print("TEST 1 — Order status + refund eligibility")
print("=" * 60)

result = support_graph.invoke({
    "messages": [HumanMessage(content="I'd like to know the status of order ORD-002 and whether I can get a refund.")],
    "user_name": "Priya",
    "account_tier": "pro",
})

print("🤖", result["messages"][-1].content)
print()

In [ ]:
# --- Test 2: Invalid order (error handling) ---

print("=" * 60)
print("TEST 2 — Invalid order ID (error recovery)")
print("=" * 60)

result = support_graph.invoke({
    "messages": [HumanMessage(content="What's the status of order ORD-999?")],
    "user_name": "Rahul",
    "account_tier": "free",
})

print("🤖", result["messages"][-1].content)
print()

In [ ]:
# --- Test 3: Escalation (Command → state mutation) ---

print("=" * 60)
print("TEST 3 — Escalation to human (Command state mutation)")
print("=" * 60)

result = support_graph.invoke({
    "messages": [HumanMessage(content="I've been waiting 2 weeks for my order ORD-001 and I'm very upset. I want to speak to a human immediately.")],
    "user_name": "Sneha",
    "account_tier": "enterprise",
})

print("🤖", result["messages"][-1].content)
print()
print("📦 State after escalation:")
print(f"   account_tier: {result['account_tier']}")  # should be 'escalated'

---
## Summary — `ToolNode` vs `create_agent`

| Feature | `create_agent` | `ToolNode` |
|---|---|---|
| Setup complexity | Low (1 call) | Medium (build a graph) |
| Control over flow | Limited | Full |
| Custom state | ❌ | ✅ |
| Parallel execution | Auto | Auto |
| Error handling | Auto | Configurable |
| Best for | Quick prototypes | Production agents |

### Key Takeaways
- ✅ `ToolNode` is the **building block** behind `create_agent` — use it for full control
- ✅ Tools return `str` → plain text, `dict` → structured data, `Command` → mutate state
- ✅ `handle_tool_errors=True` (default) lets the LLM recover from failures gracefully
- ✅ Parallel tool execution is **automatic** — no extra code needed
- ✅ Use `runtime.tool_call_id` when returning `Command` with a `ToolMessage`